In [3]:
import re
import pandas as pd
import numpy as np

In [4]:
import os

# Repo is located in Downloads folder

DATA_DIR  = os.path.join('..', 'data', 'raw')

print(f"Data dir:  {DATA_DIR}")

Data dir:  ..\data\raw


In [5]:
UCI_COLUMNS_75 = [
    'id', 'ccf', 'age', 'sex', 'painloc', 'painexer', 'relrest', 'pncaden',
    'cp', 'trestbps', 'htn', 'chol', 'smoke', 'cigs', 'years', 'fbs',
    'dm', 'famhist', 'restecg', 'ekgmo', 'ekgday', 'ekgyr', 'dig', 'prop',
    'nitr', 'pro', 'diuretic', 'proto', 'thaldur', 'thaltime', 'met',
    'thalach', 'thalrest', 'tpeakbps', 'tpeakbpd', 'dummy', 'trestbpd',
    'exang', 'xhypo', 'oldpeak', 'slope', 'rldv5', 'rldv5e', 'ca', 'restckm',
    'exerckm', 'restef', 'restwm', 'exeref', 'exerwm', 'thal', 'thalsev',
    'thalpul', 'earlobe', 'cmo', 'cday', 'cyr', 'num', 'lmt', 'ladprox',
    'laddist', 'diag', 'cxmain', 'ramus', 'om1', 'om2', 'rcaprox', 'rcadist',
    'lvx1', 'lvx2', 'lvx3', 'lvx4', 'lvf', 'cathef', 'junk'
]

In [6]:
def parse_uci_file(filepath, source_name):
    # Open the file (latin-1 handles special characters in the files)
    with open(filepath, 'r', encoding='latin-1') as f:
        content = f.read()

    # Extract all numbers and the word "name" from the raw text
    # Each patient's data ends with the word "name"
    tokens = re.findall(r'-?\d+\.?\d*|name', content)

    # Go through all tokens and collect 75 numbers before each "name"
    records = []
    i = 0
    while i < len(tokens):
        if tokens[i] == 'name':
            if i >= 75:
                records.append(tokens[i - 75 : i])  # one patient = 75 values
            i += 1
        else:
            i += 1

    # Turn the list of records into a DataFrame
    df = pd.DataFrame(records, columns=UCI_COLUMNS_75)
    df['source'] = source_name   # remember which file this came from
    return df

In [7]:
files = {
    'cleveland':     os.path.join(DATA_DIR, 'cleveland.data'),
    'hungarian':     os.path.join(DATA_DIR, 'hungarian.data'),
    'long-beach-va': os.path.join(DATA_DIR, 'long-beach-va.data'),
    'switzerland':   os.path.join(DATA_DIR, 'switzerland.data'),
    'new':           os.path.join(DATA_DIR, 'new.data'),
}

# Parse each file — this gives you 5 separate DataFrames
all_dfs = []
for name, path in files.items():
    df_temp = parse_uci_file(path, name)
    print(f"{name}: {len(df_temp)} patients")
    all_dfs.append(df_temp)

# pd.concat() stacks all 5 DataFrames on top of each other → 1 big DataFrame
combined = pd.concat(all_dfs, ignore_index=True)
#                              ↑
#     ignore_index=True resets row numbers from 0 to 1216
#     (instead of 0-292, then 0-293 again, etc.)

print(f"Total: {len(combined)} patients")

cleveland: 293 patients
hungarian: 294 patients
long-beach-va: 200 patients
switzerland: 123 patients
new: 307 patients
Total: 1217 patients


In [8]:
# The 14 key features we need + source column
KEY_COLS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
            'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num', 'source']

# Keep only those columns
df = combined[KEY_COLS].copy()

print(f"Shape before: {combined.shape}")   # should be 1217 x 76
print(f"Shape after:  {df.shape}")         # should be 1217 x 15

Shape before: (1217, 76)
Shape after:  (1217, 15)


In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [10]:
#Displaying Combined Data
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,source
0,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0,cleveland
1,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2,cleveland
2,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1,cleveland
3,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0,cleveland
4,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0,cleveland


In [11]:
print("\nData Types Before Conversion")
print(df.dtypes)

numeric_cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
                'thalach','exang','oldpeak','slope','ca','thal','num']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nData Types After Conversion:")
print(df.dtypes)


Data Types Before Conversion
age         object
sex         object
cp          object
trestbps    object
chol        object
fbs         object
restecg     object
thalach     object
exang       object
oldpeak     object
slope       object
ca          object
thal        object
num         object
source      object
dtype: object

Data Types After Conversion:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs         float64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal          int64
num           int64
source       object
dtype: object


In [12]:
df.isnull().sum().sort_values(ascending=False)

ca          1
sex         0
age         0
trestbps    0
chol        0
fbs         0
cp          0
restecg     0
thalach     0
oldpeak     0
exang       0
slope       0
thal        0
num         0
source      0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(3)

In [14]:
print(f"Rows before removing duplicates: {len(df)}")

df = df.drop_duplicates()

print(f"Rows after removing duplicates: {len(df)}")

Rows before removing duplicates: 1217
Rows after removing duplicates: 1214


In [15]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,source
0,63,1,1,145,233,1.0,2,150,0,2.3,3,0.0,6,0,cleveland
1,67,1,4,160,286,0.0,2,108,1,1.5,2,3.0,3,2,cleveland
2,67,1,4,120,229,0.0,2,129,1,2.6,2,2.0,7,1,cleveland
3,37,1,3,130,250,0.0,0,187,0,3.5,3,0.0,3,0,cleveland
4,41,0,2,130,204,0.0,2,172,0,1.4,1,0.0,3,0,cleveland


In [16]:
df[numeric_cols] = df[numeric_cols].replace(-9, np.nan)
print("Replaced -9 with NaN")

Replaced -9 with NaN


In [17]:
# Apply realistic medical bounds — anything outside these ranges is garbage
bounds = {
    'age':      (18,   120),   # human age
    'sex':      (0,   1),     # binary
    'cp':       (1,   4),     # 4 chest pain types
    'trestbps': (60,  250),   # resting blood pressure mmHg
    'chol':     (50,  700),   # cholesterol mg/dl
    'fbs':      (0,   1),     # binary
    'restecg':  (0,   2),     # 3 possible values
    'thalach':  (50,  250),   # max heart rate bpm
    'exang':    (0,   1),     # binary
    'oldpeak':  (-2,  8),     # ST depression
    'slope':    (1,   3),     # 3 slope types
    'ca':       (0,   3),     # 0-3 vessels
    'thal':     (3,   7),     # 3, 6, or 7
    'num':      (0,   4),     # diagnosis 0-4
}

for col, (lo, hi) in bounds.items():
    before = df[col].notna().sum()
    df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
    after = df[col].notna().sum()
    removed = before - after
    if removed > 0:
        print(f"  {col:12s}: {removed} garbage values removed")

print("\nDomain bounds applied!")

  age         : 313 garbage values removed
  sex         : 2 garbage values removed
  cp          : 296 garbage values removed
  trestbps    : 314 garbage values removed
  chol        : 486 garbage values removed
  fbs         : 186 garbage values removed
  restecg     : 312 garbage values removed
  thalach     : 9 garbage values removed
  exang       : 4 garbage values removed
  oldpeak     : 3 garbage values removed
  slope       : 219 garbage values removed
  ca          : 20 garbage values removed
  thal        : 321 garbage values removed
  num         : 26 garbage values removed

Domain bounds applied!


In [18]:
#Total count of NaN values in numeric columns:
print(df[numeric_cols].isnull().sum())

age         316
sex          10
cp          300
trestbps    377
chol        518
fbs         356
restecg     316
thalach     372
exang       369
oldpeak     374
slope       531
ca          627
thal        797
num          33
dtype: int64


In [19]:
df['target'] = (df['num'] > 0).astype(int)

print("=== Missing values ===")
missing = df.isnull().sum()
pct = (df.isnull().sum() / len(df) * 100).round(1)
print(pd.concat([missing, pct], axis=1, keys=['Count', '%']))

=== Missing values ===
          Count     %
age         316  26.0
sex          10   0.8
cp          300  24.7
trestbps    377  31.1
chol        518  42.7
fbs         356  29.3
restecg     316  26.0
thalach     372  30.6
exang       369  30.4
oldpeak     374  30.8
slope       531  43.7
ca          627  51.6
thal        797  65.7
num          33   2.7
source        0   0.0
target        0   0.0


In [20]:
# Fill continuous columns with MEDIAN (middle value, not affected by outliers)
median_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'slope']
for col in median_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with MODE (most common value)
mode_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal']
for col in mode_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Verify no missing values remain
print("Missing values after imputation:")
print(df.isnull().sum())

Missing values after imputation:
age          0
sex          0
cp           0
trestbps     0
chol         0
fbs          0
restecg      0
thalach      0
exang        0
oldpeak      0
slope        0
ca           0
thal         0
num         33
source       0
target       0
dtype: int64


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22792\1230364566.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_22792\1230364566.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

In [21]:
# Drop 'num' — we already extracted 'target' from it
df = df.drop(columns=['num'])

print(f"Final shape: {df.shape}")
print(f"Columns remaining: {df.columns.tolist()}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

Final shape: (1214, 15)
Columns remaining: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'source', 'target']

Missing values: 0


In [26]:
# Save cleaned data to the processed folder in the repo
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)  # creates the folder if it doesn't exist yet

output_path = os.path.join(PROCESSED_DIR, 'heart_disease_cleaned.csv')
df.to_csv(output_path, index=False)

print(f"Saved successfully!")
print(f"Location: {output_path}")
print(f"Shape: {df.shape}")

Saved successfully!
Location: ..\data\processed\heart_disease_cleaned.csv
Shape: (1214, 15)
